# Agentic AI Research Pipeline: Tool Use, Reflection & HTML Report Generation

**Author:** Sandhya [Your Last Name]  
**Date:** April 2025  
**Skills Demonstrated:** AI Agents · Tool Calling · OpenAI API · Reflective Reasoning · Agentic Pipelines

---

## 🧠 What This Project Demonstrates

This notebook showcases a **3-stage agentic AI research pipeline** I built while learning about AI agents and tool use:

| Stage | What Happens |
|-------|--------------|
| 1️⃣ **Tool Use** | Agent calls external search tools (arXiv + web) to gather real-time research data |
| 2️⃣ **Reflection** | Agent critically evaluates its own output and rewrites for quality |
| 3️⃣ **Formatting** | Final report is converted to styled HTML for sharing |

This pattern — **search → reflect → publish** — is foundational to production-grade AI agents and directly mirrors real-world systems like research assistants, competitive intelligence tools, and automated reporting pipelines.

---

## 🔗 Why This Matters for Product Managers

As a Senior Technical Product Manager working on AI platforms, understanding how agentic pipelines work under the hood helps me:
- Write sharper product requirements for AI features
- Have credible technical conversations with engineering teams
- Spot failure modes in multi-step AI workflows (hallucinations, tool errors, reasoning loops)
- Design better evaluation frameworks for AI product quality


## 📦 Setup & Imports

This pipeline uses:
- **OpenAI Python SDK** — for LLM calls with tool/function calling
- **python-dotenv** — to load API keys securely from a `.env` file (never hardcode keys!)
- **arxiv** — Python wrapper for the arXiv academic paper API
- **tavily-python** — for real-time web search results
- **IPython.display** — to render the final HTML report inline in the notebook

> ⚠️ **Before running:** Create a `.env` file in the same directory with your API keys:
> ```
> OPENAI_API_KEY=your_openai_key_here
> TAVILY_API_KEY=your_tavily_key_here
> ```
> Never commit your `.env` file to GitHub — add it to `.gitignore`!

In [ ]:
# ================================
# Standard library
# ================================
import json

# ================================
# Third-party libraries
# ================================
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import display, HTML
import arxiv
from tavily import TavilyClient

# ================================
# Environment setup
# ================================
import os
load_dotenv()  # Loads keys from .env file — keep your keys safe!

# Initialize clients
openai_client = OpenAI()  # Uses OPENAI_API_KEY from environment
tavily_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

print("✅ Environment loaded and clients initialized.")

---

## Stage 1: Tool Definitions — Giving the Agent Its Capabilities

An AI agent is only as useful as the tools it can call. Here I define two search tools:

- **`arxiv_search`** — searches academic papers from arXiv.org (great for cutting-edge AI/ML research)
- **`web_search`** — searches the live web via Tavily (great for recent news, blog posts, industry content)

### 💡 Key Concept: Tool Schemas
OpenAI's tool-calling API requires each tool to be described with a **JSON schema** — the model reads this schema to understand *what the tool does* and *what parameters to pass*. This is the mechanism that lets an LLM decide when and how to call external functions.


In [ ]:
# ================================
# Tool Implementations
# ================================

def arxiv_search(query: str, max_results: int = 3) -> list[dict]:
    """
    Search arXiv for academic papers matching the query.
    Returns a list of paper metadata dicts.
    """
    client = arxiv.Client()
    search = arxiv.Search(
        query=query,
        max_results=max_results,
        sort_by=arxiv.SortCriterion.Relevance
    )
    results = []
    for paper in client.results(search):
        results.append({
            "title": paper.title,
            "authors": [a.name for a in paper.authors],
            "published": str(paper.published.date()),
            "summary": paper.summary,
            "url": paper.entry_id,
            "pdf_url": paper.pdf_url
        })
    return results


def web_search(query: str, max_results: int = 3) -> list[dict]:
    """
    Search the web using Tavily API.
    Returns a list of result dicts with title, content, and URL.
    """
    response = tavily_client.search(query=query, max_results=max_results)
    return [
        {"title": r["title"], "content": r["content"], "url": r["url"]}
        for r in response.get("results", [])
    ]


# ================================
# Tool Schemas (for OpenAI function calling)
# ================================
# These JSON schemas tell the LLM what each tool does and how to call it.
# The model reads these descriptions to decide when to invoke a tool.

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "arxiv_search",
            "description": "Search arXiv for academic research papers on a given topic.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "The search query"},
                    "max_results": {"type": "integer", "description": "Max papers to return", "default": 3}
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "web_search",
            "description": "Search the web for current information on a topic.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "The search query"},
                    "max_results": {"type": "integer", "description": "Max results to return", "default": 3}
                },
                "required": ["query"]
            }
        }
    }
]

# Map tool names to their Python functions — used when executing tool calls
TOOL_MAP = {
    "arxiv_search": arxiv_search,
    "web_search": web_search
}

print("✅ Tools defined:", list(TOOL_MAP.keys()))

---

## Stage 2: The Research Agent — Tool-Calling Loop

This is the core of the agentic pattern. The agent:
1. Receives a research topic
2. Decides which tools to call (and with what arguments)
3. Executes those tool calls
4. Feeds results back to the LLM
5. Generates a structured research report

### 💡 Key Concept: The Tool-Calling Loop
Unlike a single prompt → response, an agent runs in a **loop**: the model can request multiple tool calls before producing its final answer. The `max_turns` parameter prevents infinite loops — an important guardrail in production systems.


In [ ]:
def run_research_agent(topic: str, max_turns: int = 5) -> str:
    """
    Run an agentic research loop on a given topic.
    
    The agent will:
    - Decide which tools to call
    - Execute those tools
    - Synthesize results into a research report
    
    Args:
        topic: The research topic to investigate
        max_turns: Maximum number of tool-calling rounds (prevents infinite loops)
    
    Returns:
        A string containing the synthesized research report
    """
    print(f"🔍 Starting research on: '{topic}'")
    
    # System prompt defines the agent's role and output expectations
    system_prompt = """You are a research assistant. Given a topic, use the available tools 
to search for relevant academic papers and web sources. Then synthesize your findings into 
a well-structured research report with:
- An introduction to the topic
- Key findings from your research
- Notable sources or studies
- Implications and future directions
- A references section

Be thorough but concise. Always cite your sources."""

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Research this topic and write a report: {topic}"}
    ]
    
    # Agentic loop: keep going until the model stops calling tools
    for turn in range(max_turns):
        print(f"  Turn {turn + 1}/{max_turns}...")
        
        response = openai_client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=TOOLS,
            tool_choice="auto"  # Let the model decide when to call tools
        )
        
        message = response.choices[0].message
        messages.append(message)  # Add assistant turn to history
        
        # If no tool calls, the agent is done — return the final text
        if not message.tool_calls:
            print("✅ Agent completed research.")
            return message.content
        
        # Execute each tool call and feed results back to the model
        for tool_call in message.tool_calls:
            tool_name = tool_call.function.name
            tool_args = json.loads(tool_call.function.arguments)
            
            print(f"  🛠️  Calling tool: {tool_name}({tool_args})")
            
            # Execute the tool
            tool_fn = TOOL_MAP.get(tool_name)
            if tool_fn:
                result = tool_fn(**tool_args)
            else:
                result = {"error": f"Tool '{tool_name}' not found"}
            
            # Add tool result to conversation history
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps(result)
            })
    
    # If we hit max_turns, return whatever the last text response was
    print("⚠️  Reached max_turns limit.")
    last_text = next(
        (m.content for m in reversed(messages) if hasattr(m, 'content') and m.content),
        "No report generated."
    )
    return last_text

print("✅ Research agent defined.")

---

## Stage 3: Reflection — The Agent Critiques Its Own Work

Reflection is what separates a basic LLM call from a thoughtful agent. After generating a first draft, the agent:
1. Reviews its own report
2. Identifies **strengths, limitations, gaps, and opportunities**
3. Rewrites the report incorporating the critique

### 💡 Key Concept: Self-Reflection in AI Agents
This mirrors how skilled professionals work — write a draft, critique it, revise. In production AI systems, reflection loops significantly improve output quality. This is the core idea behind frameworks like **Reflexion** (Shinn et al., 2023) and is used in advanced agents like AutoGPT and LangGraph workflows.


In [ ]:
def reflect_and_revise(report: str) -> dict:
    """
    Takes a research report, critiques it, and produces a revised version.
    
    This implements the 'reflection' pattern in agentic AI systems:
    the model acts as its own critic, then rewrites based on that critique.
    
    Args:
        report: The preliminary research report text
    
    Returns:
        dict with 'reflection' (critique) and 'revised_report' (improved version)
    """
    print("🪞 Running reflection and revision...")
    
    reflection_prompt = f"""You are a critical editor reviewing a research report.

Here is the report to review:
---
{report}
---

Please do two things:

1. REFLECTION: Critique the report by identifying:
   - Strengths: What the report does well
   - Limitations: What's missing or weak
   - Suggestions: Specific improvements to make
   - Opportunities: Ways to add depth or expand the analysis

2. REVISED REPORT: Rewrite the report incorporating your critique.

Format your response as JSON with exactly these keys:
  - "reflection": your critique (string)
  - "revised_report": the improved report (string)
"""
    
    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": reflection_prompt}],
        response_format={"type": "json_object"}  # Force JSON output
    )
    
    result = json.loads(response.choices[0].message.content)
    print("✅ Reflection complete.")
    return result


print("✅ Reflection function defined.")

---

## Stage 4: HTML Report Generation

The final stage converts the Markdown research report into a **styled HTML page** — ready to share, embed in a dashboard, or publish to the web.

This step demonstrates how AI agents can handle **output formatting as a distinct task**, separating content generation from presentation — a clean design pattern for production pipelines.


In [ ]:
def convert_to_html(report: str) -> str:
    """
    Convert a plain-text research report into a styled HTML page.
    
    Args:
        report: The final research report text (Markdown or plain text)
    
    Returns:
        A complete HTML string with inline CSS styling
    """
    print("📄 Converting report to HTML...")
    
    html_prompt = f"""Convert the following research report into a clean, well-styled HTML page.

Requirements:
- Use semantic HTML (h1, h2, h3, p, ul, a tags appropriately)
- Include inline CSS in a <style> block for clean typography
- Make links clickable with target="_blank"
- Use a clean, professional color scheme (blues and grays work well)
- Include a <title> tag matching the report topic
- Return ONLY the HTML — no explanations, no markdown fences

Report to convert:
---
{report}
---
"""
    
    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": html_prompt}]
    )
    
    html = response.choices[0].message.content.strip()
    # Clean up any accidental markdown code fences
    html = html.replace("```html", "").replace("```", "").strip()
    
    print("✅ HTML conversion complete.")
    return html


print("✅ HTML converter defined.")

---

## 🚀 Run the Full Pipeline

Now let's put it all together. Change `RESEARCH_TOPIC` to any topic you're curious about!

The pipeline will:
1. Search arXiv and the web for relevant content
2. Generate a research report
3. Reflect on and revise the report
4. Convert it to styled HTML


In [ ]:
# ============================
# 👇 Change this to your topic
# ============================
RESEARCH_TOPIC = "multimodal large language models in healthcare"

# ============================
# Stage 1: Research with tools
# ============================
print("=" * 60)
print("STAGE 1: Generating research report with tool use")
print("=" * 60)
preliminary_report = run_research_agent(RESEARCH_TOPIC)
print("\n--- Preliminary Report (first 500 chars) ---")
print(preliminary_report[:500], "...\n")

In [ ]:
# ============================
# Stage 2: Reflection & Revision
# ============================
print("=" * 60)
print("STAGE 2: Reflection and revision")
print("=" * 60)
reflection_output = reflect_and_revise(preliminary_report)

print("\n--- Agent's Self-Critique ---")
print(reflection_output.get("reflection", ""))

print("\n--- Revised Report (first 500 chars) ---")
print(reflection_output.get("revised_report", "")[:500], "...\n")

In [ ]:
# ============================
# Stage 3: HTML Output
# ============================
print("=" * 60)
print("STAGE 3: Converting to styled HTML")
print("=" * 60)

final_report = reflection_output.get("revised_report", preliminary_report)
html_output = convert_to_html(final_report)

# Save HTML to file
output_filename = "research_report.html"
with open(output_filename, "w", encoding="utf-8") as f:
    f.write(html_output)
print(f"💾 Report saved to: {output_filename}")

# Display inline in notebook
print("\n--- Rendering HTML Report ---")
display(HTML(html_output))

---

## 📊 Key Takeaways

### What I Built
A **3-stage agentic pipeline** that autonomously researches a topic, critiques its own output, and publishes a formatted report.

### Patterns I Applied

| Pattern | What It Is | Why It Matters |
|---------|------------|----------------|
| **Tool Use** | LLM calls external APIs mid-conversation | Grounds AI in real data, reduces hallucinations |
| **Agentic Loop** | Model runs multiple turns until task complete | Enables complex, multi-step reasoning |
| **Self-Reflection** | Model critiques and rewrites its own output | Significantly improves output quality |
| **Structured Outputs** | JSON response format for predictable parsing | Essential for production pipelines |
| **Secure Secrets** | API keys in `.env`, never hardcoded | Security best practice |

### What I'd Explore Next
- Add **memory** so the agent can build on prior research sessions
- Implement **parallel tool calling** for faster research gathering
- Add a **confidence score** — agent flags when it's uncertain about claims
- Integrate with a vector database for **RAG** (retrieval-augmented generation)
- Build a **multi-agent version** where one agent researches and another fact-checks

---

## 🔧 How to Run This Yourself

1. Clone this repo
2. Install dependencies: `pip install openai tavily-python arxiv python-dotenv`
3. Create a `.env` file with your API keys (see Setup section above)
4. Run cells top to bottom in Jupyter
5. Change `RESEARCH_TOPIC` to anything you want to research!

---

*Built as part of my AI learning journey — exploring agentic AI systems to deepen my technical foundation as a Senior Technical Product Manager.*
